# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path
import polars as pl
import duckdb

DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_PATH = DATA_DIR / "flood_dataset.parquet"

lf = pl.scan_parquet(FILE_PATH)

print("Dataset berhasil dimuat.")
print("Jumlah baris:", lf.select(pl.len()).collect().item())

Dataset berhasil dimuat.
Jumlah baris: 1152432


In [2]:
duplicate_keys = (
    lf.group_by(["district_id", "date"])
    .agg(pl.len().alias("jumlah"))
    .filter(pl.col("jumlah") > 1)
    .collect()
)

print("Jumlah pasangan kecamatan-tanggal duplikat:",
      duplicate_keys.height)

display(duplicate_keys.head(10))

Jumlah pasangan kecamatan-tanggal duplikat: 0


district_id,date,jumlah
str,datetime[μs],u32


In [3]:
negative_check = lf.select([
    (pl.col("rain_1d") < 0).sum().alias("rain_1d_negatif"),
    (pl.col("rain_3d") < 0).sum().alias("rain_3d_negatif"),
    (pl.col("rain_7d") < 0).sum().alias("rain_7d_negatif"),
    (pl.col("rain_30d") < 0).sum().alias("rain_30d_negatif"),
    (pl.col("disc_now") < 0).sum().alias("disc_now_negatif"),
    (pl.col("disc_7d_mean") < 0).sum().alias("disc_7d_mean_negatif")
]).collect()

display(negative_check)

rain_1d_negatif,rain_3d_negatif,rain_7d_negatif,rain_30d_negatif,disc_now_negatif,disc_7d_mean_negatif
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [4]:
date_check = lf.select([
    (pl.col("date").dt.year() != pl.col("year"))
    .sum()
    .alias("tahun_tidak_sesuai"),

    (pl.col("date").dt.month() != pl.col("month"))
    .sum()
    .alias("bulan_tidak_sesuai")
]).collect()

display(date_check)

tahun_tidak_sesuai,bulan_tidak_sesuai
u32,u32
0,0


In [8]:
label_check = lf.select([
    (
        (pl.col("flood_label") != 0.0) &
        (pl.col("flood_label") != 1.0)
    ).sum().alias("label_tidak_valid"),

    (
        (pl.col("coastal") != 0) &
        (pl.col("coastal") != 1)
    ).sum().alias("coastal_tidak_valid")
]).collect()

display(label_check)

label_tidak_valid,coastal_tidak_valid
u32,u32
0,0


In [9]:
rain_check = lf.select([
    (pl.col("rain_3d") < pl.col("rain_1d"))
    .sum()
    .alias("rain_3d_di_bawah_rain_1d"),

    (pl.col("rain_7d") < pl.col("rain_3d"))
    .sum()
    .alias("rain_7d_di_bawah_rain_3d"),

    (pl.col("rain_30d") < pl.col("rain_7d"))
    .sum()
    .alias("rain_30d_di_bawah_rain_7d")
]).collect()

display(rain_check)

rain_3d_di_bawah_rain_1d,rain_7d_di_bawah_rain_3d,rain_30d_di_bawah_rain_7d
u32,u32,u32
12703,23266,693


In [10]:
duplicate_keys = (
    lf.group_by(["district_id", "date"])
    .agg(pl.len().alias("jumlah"))
    .filter(pl.col("jumlah") > 1)
    .collect()
)

print("Duplikasi kecamatan-tanggal:", duplicate_keys.height)
display(duplicate_keys.head(10))

Duplikasi kecamatan-tanggal: 0


district_id,date,jumlah
str,datetime[μs],u32


In [11]:
negative_check = lf.select([
    (pl.col("rain_1d") < 0).sum().alias("rain_1d_negatif"),
    (pl.col("rain_3d") < 0).sum().alias("rain_3d_negatif"),
    (pl.col("rain_7d") < 0).sum().alias("rain_7d_negatif"),
    (pl.col("rain_30d") < 0).sum().alias("rain_30d_negatif"),
    (pl.col("disc_now") < 0).sum().alias("disc_now_negatif"),
    (pl.col("disc_7d_mean") < 0).sum().alias("disc_7d_mean_negatif")
]).collect()

display(negative_check)

rain_1d_negatif,rain_3d_negatif,rain_7d_negatif,rain_30d_negatif,disc_now_negatif,disc_7d_mean_negatif
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [12]:
date_check = lf.select([
    (pl.col("date").dt.year() != pl.col("year"))
    .sum().alias("tahun_tidak_sesuai"),

    (pl.col("date").dt.month() != pl.col("month"))
    .sum().alias("bulan_tidak_sesuai")
]).collect()

display(date_check)

tahun_tidak_sesuai,bulan_tidak_sesuai
u32,u32
0,0


In [13]:
rain_check = lf.select([
    (pl.col("rain_3d") < pl.col("rain_1d"))
    .sum().alias("rain_3d_di_bawah_rain_1d"),

    (pl.col("rain_7d") < pl.col("rain_3d"))
    .sum().alias("rain_7d_di_bawah_rain_3d"),

    (pl.col("rain_30d") < pl.col("rain_7d"))
    .sum().alias("rain_30d_di_bawah_rain_7d")
]).collect()

display(rain_check)

rain_3d_di_bawah_rain_1d,rain_7d_di_bawah_rain_3d,rain_30d_di_bawah_rain_7d
u32,u32,u32
12703,23266,693


In [14]:
numeric_columns = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d"
]

outlier_results = []

for column in numeric_columns:
    stats = lf.select([
        pl.col(column).quantile(0.25).alias("q1"),
        pl.col(column).quantile(0.75).alias("q3")
    ]).collect()

    q1 = stats["q1"][0]
    q3 = stats["q3"][0]

    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = (
        lf.filter(
            (pl.col(column) < lower) |
            (pl.col(column) > upper)
        )
        .select(pl.len())
        .collect()
        .item()
    )

    outlier_results.append({
        "kolom": column,
        "batas_bawah": lower,
        "batas_atas": upper,
        "jumlah_outlier": count
    })

outlier_df = pl.DataFrame(outlier_results)
display(outlier_df)

kolom,batas_bawah,batas_atas,jumlah_outlier
str,f64,f64,i64
"""rain_1d""",-13.95,24.05,60717
"""rain_3d""",-40.85,73.95,34406
"""rain_7d""",-87.3,167.9,23955
"""rain_30d""",-315.3,685.5,10907
"""disc_now""",-14.16,25.52,159404
"""disc_7d_mean""",-14.305,25.883571,158128
"""disc_change_3d""",-2.015,1.705,302722


In [15]:
con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE VIEW flood_data AS
    SELECT *
    FROM read_parquet('{FILE_PATH.as_posix()}')
""")

quality_summary = con.execute("""
    SUMMARIZE flood_data
""").df()

display(quality_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,district_id,VARCHAR,IDN.10.13.10_1,IDN.9.9.9_1,316,NaN,NaN,NaN,NaN,NaN,1152432,0.0
1,date,TIMESTAMP,2015-01-30 00:00:00,2024-12-31 00:00:00,3712,2020-01-15 12:00:00,NaN,2017-07-23 03:35:52.340014,2020-01-15 22:21:26.672156,2022-07-09 02:44:46.356104,1152432,0.0
2,year,INTEGER,2015,2024,10,2019.5364238410596,2.8556717940747394,2017,2020,2022,1152432,0.0
3,rain_1d,DOUBLE,0.0,258.7,1332,6.793897687671181,10.10181064199661,0.3102058265651896,2.9782538095282907,9.777347223340776,1152432,0.0
4,rain_3d,DOUBLE,0.0,415.70000000000005,54437,20.382071566912654,23.51372439659673,2.201293751937434,13.672714821062801,30.881690418216024,1152432,0.0
5,rain_7d,DOUBLE,0.0,645.2,43777,47.561081174425716,46.715256295523545,8.41853931595645,37.61793087214816,72.18495608160401,1152432,0.0
6,rain_30d,DOUBLE,0.0,1502.5,30705,203.64353714578917,165.6060230160802,60.07664049156212,179.39753267769353,310.20342533246475,1152432,0.0
7,disc_now,DOUBLE,0.0,1467.93,2863,20.809406672146086,71.05971098343126,0.7225371787735888,2.7145676263902923,10.669058074500233,1152432,0.0
8,disc_7d_mean,DOUBLE,0.0,1307.3185714285714,183910,20.806564402186886,70.15783131833687,0.7626605869124681,2.7895883919333695,10.820594251990162,1152432,0.0
9,disc_change_3d,DOUBLE,-758.7300000000001,1076.38,53862,0.0009839279020340383,18.861786504809054,-0.6191748743394475,-0.03021803170823922,0.3096824373378738,1152432,0.0


In [16]:
rain_inconsistent = (
    lf.filter(
        (pl.col("rain_3d") < pl.col("rain_1d")) |
        (pl.col("rain_7d") < pl.col("rain_3d")) |
        (pl.col("rain_30d") < pl.col("rain_7d"))
    )
    .select([
        "district_id",
        "date",
        "rain_1d",
        "rain_3d",
        "rain_7d",
        "rain_30d"
    ])
    .head(20)
    .collect()
)

display(rain_inconsistent)

district_id,date,rain_1d,rain_3d,rain_7d,rain_30d
str,datetime[μs],f64,f64,f64,f64
"""IDN.10.13.10_1""",2015-05-23 00:00:00,0.3,0.3,0.3,199.0
"""IDN.10.13.10_1""",2015-05-24 00:00:00,0.2,0.5,0.5,169.7
"""IDN.10.13.10_1""",2015-05-25 00:00:00,0.0,0.5,0.5,153.1
"""IDN.10.13.10_1""",2015-06-20 00:00:00,0.3,0.3,0.3,28.8
"""IDN.10.13.10_1""",2015-06-21 00:00:00,0.4,0.7,0.7,29.2
…,…,…,…,…,…
"""IDN.10.13.10_1""",2015-07-31 00:00:00,0.9,1.2,1.2,8.7
"""IDN.10.13.10_1""",2015-08-01 00:00:00,0.9,2.1,2.1,8.8
"""IDN.10.13.10_1""",2015-08-28 00:00:00,0.1,0.1,0.1,11.7


In [17]:
outlier_detail = []

for column in numeric_columns:
    stats = lf.select([
        pl.col(column).quantile(0.25).alias("q1"),
        pl.col(column).quantile(0.75).alias("q3")
    ]).collect()

    q1 = stats["q1"][0]
    q3 = stats["q3"][0]

    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    counts = lf.select([
        (pl.col(column) < lower).sum().alias("di_bawah"),
        (pl.col(column) > upper).sum().alias("di_atas")
    ]).collect()

    outlier_detail.append({
        "kolom": column,
        "batas_bawah": lower,
        "batas_atas": upper,
        "outlier_bawah": counts["di_bawah"][0],
        "outlier_atas": counts["di_atas"][0]
    })

display(pl.DataFrame(outlier_detail))

kolom,batas_bawah,batas_atas,outlier_bawah,outlier_atas
str,f64,f64,i64,i64
"""rain_1d""",-13.95,24.05,0,60717
"""rain_3d""",-40.85,73.95,0,34406
"""rain_7d""",-87.3,167.9,0,23955
"""rain_30d""",-315.3,685.5,0,10907
"""disc_now""",-14.16,25.52,0,159404
"""disc_7d_mean""",-14.305,25.883571,0,158128
"""disc_change_3d""",-2.015,1.705,157001,145721


In [18]:
check_3d = (
    lf.filter(pl.col("rain_3d") < pl.col("rain_1d"))
    .select([
        "district_id", "date",
        "rain_1d", "rain_3d"
    ])
    .head(15)
    .collect()
)

display(check_3d)

district_id,date,rain_1d,rain_3d
str,datetime[μs],f64,f64
"""IDN.10.13.10_1""",2022-04-24 00:00:00,5.2,5.2
"""IDN.10.13.10_1""",2022-05-29 00:00:00,1.1,1.1
"""IDN.10.13.10_1""",2022-07-01 00:00:00,0.3,0.3
"""IDN.10.13.10_1""",2022-07-11 00:00:00,2.2,2.2
"""IDN.10.13.10_1""",2022-07-28 00:00:00,0.7,0.7
…,…,…,…
"""IDN.10.13.10_1""",2023-05-30 00:00:00,0.1,0.1
"""IDN.10.13.10_1""",2023-06-03 00:00:00,0.3,0.3
"""IDN.10.13.10_1""",2023-06-24 00:00:00,0.2,0.2


In [19]:
check_7d = (
    lf.filter(pl.col("rain_7d") < pl.col("rain_3d"))
    .select([
        "district_id", "date",
        "rain_3d", "rain_7d"
    ])
    .head(15)
    .collect()
)

display(check_7d)

district_id,date,rain_3d,rain_7d
str,datetime[μs],f64,f64
"""IDN.10.13.10_1""",2015-05-23 00:00:00,0.3,0.3
"""IDN.10.13.10_1""",2015-05-24 00:00:00,0.5,0.5
"""IDN.10.13.10_1""",2015-05-25 00:00:00,0.5,0.5
"""IDN.10.13.10_1""",2015-06-20 00:00:00,0.3,0.3
"""IDN.10.13.10_1""",2015-06-21 00:00:00,0.7,0.7
…,…,…,…
"""IDN.10.13.10_1""",2015-07-10 00:00:00,4.3,4.3
"""IDN.10.13.10_1""",2015-07-20 00:00:00,0.1,0.1
"""IDN.10.13.10_1""",2015-07-21 00:00:00,1.0,1.0


In [20]:
check_30d = (
    lf.filter(pl.col("rain_30d") < pl.col("rain_7d"))
    .select([
        "district_id", "date",
        "rain_7d", "rain_30d"
    ])
    .head(15)
    .collect()
)

display(check_30d)

district_id,date,rain_7d,rain_30d
str,datetime[μs],f64,f64
"""IDN.10.28.10_1""",2023-08-23 00:00:00,0.2,0.2
"""IDN.10.28.10_1""",2023-08-24 00:00:00,0.2,0.2
"""IDN.10.28.10_1""",2023-08-25 00:00:00,0.2,0.2
"""IDN.10.28.10_1""",2023-08-26 00:00:00,0.2,0.2
"""IDN.10.28.10_1""",2023-08-27 00:00:00,0.2,0.2
…,…,…,…
"""IDN.10.28.2_1""",2023-09-14 00:00:00,0.4,0.4
"""IDN.10.28.2_1""",2023-09-15 00:00:00,0.4,0.4
"""IDN.10.28.2_1""",2023-09-16 00:00:00,1.2,1.2


In [21]:
display(
    lf.select([
        pl.col("rain_1d").min().alias("rain_1d_min"),
        pl.col("rain_1d").max().alias("rain_1d_max"),

        pl.col("rain_3d").min().alias("rain_3d_min"),
        pl.col("rain_3d").max().alias("rain_3d_max"),

        pl.col("rain_7d").min().alias("rain_7d_min"),
        pl.col("rain_7d").max().alias("rain_7d_max"),

        pl.col("rain_30d").min().alias("rain_30d_min"),
        pl.col("rain_30d").max().alias("rain_30d_max"),

        pl.col("disc_now").min().alias("disc_now_min"),
        pl.col("disc_now").max().alias("disc_now_max"),

        pl.col("disc_change_3d").min().alias("disc_change_3d_min"),
        pl.col("disc_change_3d").max().alias("disc_change_3d_max")
    ]).collect()
)

rain_1d_min,rain_1d_max,rain_3d_min,rain_3d_max,rain_7d_min,rain_7d_max,rain_30d_min,rain_30d_max,disc_now_min,disc_now_max,disc_change_3d_min,disc_change_3d_max
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,258.7,0.0,415.7,0.0,645.2,0.0,1502.5,0.0,1467.93,-758.73,1076.38


In [22]:
# Cleaning dataset dengan Polars LazyFrame

cleaned_lf = (
    lf
    .with_columns([
        pl.col("district_id").cast(pl.String),
        pl.col("date").cast(pl.Datetime),
        pl.col("year").cast(pl.Int64),
        pl.col("month").cast(pl.Int64),
    ])
)

# Simpan hasil cleaning ke format Parquet
cleaned_path = OUTPUT_DIR / "flood_cleaned.parquet"

cleaned_lf.sink_parquet(cleaned_path)

print("Dataset hasil cleaning berhasil disimpan:", cleaned_path)

Dataset hasil cleaning berhasil disimpan: ../data/processed/flood_cleaned.parquet


In [23]:
cleaned_check = pl.scan_parquet(cleaned_path)

print("Jumlah baris sebelum cleaning:", lf.select(pl.len()).collect().item())
print("Jumlah baris sesudah cleaning:", cleaned_check.select(pl.len()).collect().item())
print("Jumlah kolom sesudah cleaning:", len(cleaned_check.collect_schema()))

display(cleaned_check.head(5).collect())

Jumlah baris sebelum cleaning: 1152432
Jumlah baris sesudah cleaning: 1152432
Jumlah kolom sesudah cleaning: 13


district_id,date,year,rain_1d,rain_3d,rain_7d,rain_30d,disc_now,disc_7d_mean,disc_change_3d,month,coastal,flood_label
str,datetime[μs],i64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64
"""IDN.10.13.10_1""",2015-01-30 00:00:00,2015,13.3,35.0,62.0,348.3,25.98,26.238571,0.56,1,1,0.0
"""IDN.10.13.10_1""",2015-01-31 00:00:00,2015,12.5,44.0,67.2,348.1,26.35,25.92,2.35,1,1,0.0
"""IDN.10.13.10_1""",2015-02-01 00:00:00,2015,10.4,36.2,67.0,352.5,25.98,25.548571,1.8,2,1,0.0
"""IDN.10.13.10_1""",2015-02-02 00:00:00,2015,1.9,24.8,65.9,326.8,24.76,25.238571,-1.22,2,1,0.0
"""IDN.10.13.10_1""",2015-02-03 00:00:00,2015,20.7,33.0,80.5,327.8,25.98,25.318571,-0.37,2,1,0.0


In [24]:
# Periksa jumlah nilai null pada setiap kolom
null_check = cleaned_check.select([
    pl.col(col).null_count().alias(col)
    for col in cleaned_check.collect_schema().names()
]).collect()

display(null_check)


district_id,date,year,rain_1d,rain_3d,rain_7d,rain_30d,disc_now,disc_7d_mean,disc_change_3d,month,coastal,flood_label
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0


In [25]:
rows_with_null = (
    cleaned_check
    .filter(
        pl.any_horizontal([
            pl.col(col).is_null()
            for col in cleaned_check.collect_schema().names()
        ])
    )
    .select(["district_id", "date", "rain_1d", "rain_3d",
             "rain_7d", "rain_30d", "disc_now",
             "disc_7d_mean", "disc_change_3d",
             "month", "coastal", "flood_label"])
    .head(10)
    .collect()
)

display(rows_with_null)

district_id,date,rain_1d,rain_3d,rain_7d,rain_30d,disc_now,disc_7d_mean,disc_change_3d,month,coastal,flood_label
str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,i64,i64,f64


In [26]:
import duckdb

con = duckdb.connect()

cleaned_path_sql = cleaned_path.as_posix()

con.execute(f"""
    CREATE OR REPLACE VIEW flood_cleaned AS
    SELECT *
    FROM read_parquet('{cleaned_path_sql}')
""")

summary_cleaned = con.execute("""
    SUMMARIZE flood_cleaned
""").df()

display(summary_cleaned)

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,district_id,VARCHAR,IDN.10.13.10_1,IDN.9.9.9_1,316,NaN,NaN,NaN,NaN,NaN,1152432,0.0
1,date,TIMESTAMP,2015-01-30 00:00:00,2024-12-31 00:00:00,3712,2020-01-15 12:00:00,NaN,2017-07-24 19:20:04.641476,2020-01-14 00:07:34.356587,2022-07-08 12:44:15.190312,1152432,0.0
2,year,BIGINT,2015,2024,10,2019.5364238410596,2.855671794074755,2017,2020,2022,1152432,0.0
3,rain_1d,DOUBLE,0.0,258.7,1332,6.793897687672356,10.10181064199664,0.2999066901667544,2.980851289565543,9.771874988204278,1152432,0.0
4,rain_3d,DOUBLE,0.0,415.70000000000005,54437,20.382071566912874,23.51372439659689,2.2082474919506114,13.67977132355138,30.91154949330336,1152432,0.0
5,rain_7d,DOUBLE,0.0,645.2,43777,47.561081174420856,46.71525629552395,8.428560903025264,37.641343999975156,72.1583484578544,1152432,0.0
6,rain_30d,DOUBLE,0.0,1502.5,30705,203.64353714579474,165.60602301607756,60.3077156562032,179.67410183617298,310.1630055256878,1152432,0.0
7,disc_now,DOUBLE,0.0,1467.93,2863,20.80940667215142,71.05971098343194,0.7273483827001163,2.7092800746593397,10.655295608178413,1152432,0.0
8,disc_7d_mean,DOUBLE,0.0,1307.3185714285714,183910,20.806564402188368,70.15783131833689,0.766527581097045,2.78505489836238,10.847181501525418,1152432,0.0
9,disc_change_3d,DOUBLE,-758.7300000000001,1076.38,53862,0.000983927902036813,18.86178650480913,-0.6222873650595675,-0.031153293462074762,0.30740124188753976,1152432,0.0


In [27]:
row_count = con.execute("""
    SELECT COUNT(*)
    FROM flood_cleaned
""").fetchone()[0]

print("Jumlah baris berdasarkan DuckDB:", row_count)

Jumlah baris berdasarkan DuckDB: 1152432


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [ ]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [ ]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [ ]:
# TODO: sesuaikan nama kolom dengan dataset Anda
df_missing_handled = df.with_columns([
    # Contoh: isi kolom numerik dengan median
    # pl.col('kolom_numerik').fill_null(pl.col('kolom_numerik').median()),

    # Contoh: isi kolom kategorik dengan nilai default
    # pl.col('kolom_kategorik').fill_null('tidak diketahui'),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')

## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [ ]:
# TODO: ganti 'kolom_numerik' dengan kolom numerik utama pada dataset Anda
KOLOM_NUMERIK = 'kolom_numerik'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Windsorization: clip nilai ke batas yang valid, bukan menghapus barisnya
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)

## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [ ]:
# TODO: ganti 'id_unik' dengan kolom kunci unik pada dataset Anda
KOLOM_KUNCI = 'id_unik'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [ ]:
connection.register('dataset_bersih', df_dedup)

# TODO: sesuaikan aturan validasi dengan kolom dan batas nilai pada dataset Anda
validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas
    FROM dataset_bersih
""").pl()
validasi

## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

## Dokumentasi Milestone 2 – Data Cleaning dan Data Quality

### 1. Dimensi Kualitas Data yang Paling Bermasalah

Berdasarkan hasil profiling dan validasi, dimensi kualitas data yang paling perlu diperhatikan adalah **konsistensi (consistency)** dan **akurasi (accuracy)**.

Beberapa temuan:
- Ditemukan nilai ekstrem pada beberapa variabel berdasarkan metode Interquartile Range (IQR).
- Ditemukan observasi dengan nilai curah hujan antarperiode yang tidak selalu berurutan secara numerik, misalnya `rain_3d` lebih kecil daripada `rain_1d`.
- Nilai ekstrem dan perbedaan antarperiode tersebut belum dapat dipastikan sebagai kesalahan karena definisi dan karakteristik variabel perlu diperhatikan.

Sementara itu, kelengkapan (completeness) dan keunikan (uniqueness) data tergolong baik karena tidak ditemukan missing values maupun duplikasi.

### 2. Strategi Penanganan Missing Values

Hasil pemeriksaan menunjukkan bahwa seluruh 13 kolom memiliki 0 missing values.

Strategi yang dipilih adalah **tidak melakukan imputasi atau penghapusan data**, karena tidak terdapat nilai kosong yang perlu ditangani.

### 3. Strategi Penanganan Outlier

Pemeriksaan outlier dilakukan menggunakan metode Interquartile Range (IQR), dengan batas:

- Batas bawah = Q1 − 1,5 × IQR
- Batas atas = Q3 + 1,5 × IQR

Beberapa variabel memiliki nilai di luar batas IQR. Namun, nilai tersebut tidak langsung dihapus atau diubah karena outlier statistik belum tentu merupakan kesalahan pengukuran.

Strategi yang dipilih adalah mempertahankan nilai ekstrem yang ditemukan dan mendokumentasikannya untuk menghindari hilangnya informasi penting, terutama karena curah hujan dan indikator hidrologi dapat memiliki variasi yang besar.

### 4. Jumlah Baris yang Berubah

| Keterangan | Jumlah Baris |
|---|---:|
| Sebelum cleaning | 1.152.432 |
| Sesudah cleaning | 1.152.432 |
| Baris yang dihapus | 0 |
| Perubahan jumlah baris | 0 |

Tidak ada baris yang dihapus karena hasil validasi tidak menunjukkan adanya data yang terbukti tidak valid. Proses cleaning dilakukan dengan penyesuaian tipe data dan penyimpanan ulang ke format Parquet.

### 5. Aturan Validasi yang Diterapkan

Aturan validasi yang digunakan meliputi:

1. **Missing values:** memastikan seluruh kolom tidak memiliki nilai null.
2. **Duplikasi baris:** memastikan tidak terdapat baris duplikat.
3. **Keunikan observasi:** memastikan kombinasi `district_id` dan `date` tidak duplikat.
4. **Nilai numerik:** memeriksa nilai negatif pada variabel `rain_1d`, `rain_3d`, `rain_7d`, `rain_30d`, `disc_now`, dan `disc_7d_mean`.
5. **Konsistensi waktu:** memastikan nilai `year` dan `month` sesuai dengan kolom `date`.
6. **Validitas label:** memastikan `flood_label` hanya bernilai 0 atau 1.
7. **Validitas atribut pesisir:** memastikan `coastal` hanya bernilai 0 atau 1.
8. **Outlier:** mengidentifikasi nilai ekstrem menggunakan metode IQR tanpa menghapusnya secara otomatis.

Hasil validasi menunjukkan tidak ditemukan null, duplikasi, nilai negatif pada variabel yang diperiksa, ketidaksesuaian waktu, maupun nilai kategori yang tidak valid.

### 6. Hasil Akhir Cleaning

Data dibersihkan menggunakan Polars dengan pendekatan Lazy Evaluation (`scan_parquet`) dan disimpan dalam format Parquet sebagai:

`data/processed/flood_cleaned.parquet`

Selanjutnya, DuckDB digunakan untuk melakukan profiling akhir dengan perintah `SUMMARIZE`. Hasilnya menunjukkan jumlah baris tetap 1.152.432, jumlah kolom 13, dan persentase null 0% pada seluruh kolom.

Dengan demikian, proses cleaning mempertahankan seluruh observasi karena tidak ditemukan alasan yang cukup kuat untuk menghapus data. Dataset hasil cleaning siap digunakan untuk analisis lebih lanjut pada Milestone 3.